In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import datetime as dt
import utils.helper as helper
import utils.eval as eval

import os
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import importlib
import utils.helper as helper
importlib.reload(helper)

### load incidence and meta data from converted rds files

In [ ]:
data_path = "data/2_processed/covid_incidence.csv"
incidence = pd.read_csv(data_path)

model = "arima"
state = "az"
data_path = f"data/3_incidence predictions/{model}/"

arr = np.load(f"{data_path}{state}_forecasts.npy")
meta = np.load(f"{data_path}{state}_meta.npz")
start_dates_days = meta["start_dates"]
dates_days = meta["dates"]

# convert int days since 1970-01-01 to datetime.date
to_date = lambda d: dt.date(1970, 1, 1) + dt.timedelta(days=int(d))
start_dates = np.array([to_date(d) for d in start_dates_days])
start_dates

## Peak time forecast generation

In [ ]:
# set parameter packs
states = ["az","ca","il","md","nj","ny"]
models = ["arima","gp","prophet","deepar","chronos2"]

P_window = {"window_len": 14, "stride_size": 14, "n_windows": 51}
P_peak   = {"peak_window": 11, "rel_thr": 0.05}
P_ma     = {"ma": 7}
qt_levels = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95], dtype=np.float64)
seed = 123

start_indices = helper.get_window_start_indices_from_dates(incidence.index, start_dates)

In [ ]:
for state in states:
    for model in models:
        y_obs = incidence[[state]].to_numpy()[:, 0]              # (n_rows,)

        # Supplement observed context for each window
        obs_win = helper.split_to_supple(y_obs, start_indices, P_peak)  # (W, half+6, 1)

        # Build long observed peaks (0/1) once, from the entire series (vectorized)
        half = (P_peak["peak_window"] - 1) // 2       # 5
        head_drop = half + 6                          # 11
        tail_drop = half                              # 5

        y2d = incidence[[state]].to_numpy().astype(np.float32)
        peaks_core = helper.compute_2d_inci_peak(y2d, P_peak).astype(np.uint8)  # (n_rows - head_drop - tail_drop, 1)

        ## --- 2) place core back into a full-length daily array (zeros elsewhere) ---
        n_rows = y2d.shape[0]
        peaks_full = np.zeros(n_rows, dtype=np.uint8)
        peaks_full[head_drop : n_rows - tail_drop] = peaks_core.flatten()  # aligned to original index

        ## --- 3) slice exactly the forecast period you care about ---
        t0 = int(start_indices[0])                              # first window start (e.g., 2020-03-15)
        t_end = int(start_indices[-1]) + P_window["window_len"] # last window end is exclusive (… +14)

        peaks_forecast_span = peaks_full[t0:t_end]              # 1D array length = (t_end - t0)

        ## --- 4) attach dates (long format, no windowing) ---
        gt_peak_df = pd.DataFrame(
            {'peak': peaks_forecast_span.astype(np.uint8)},
            index=incidence.index[t0:t_end]
        )
        gt_peak_df.to_csv(f"data/5_test data/{state}_test.csv")

        # Process the model forecasts
        forecast = np.load(f"data/3_incidence predictions/{model}/{state}_forecasts.npy")
        if model == "chronos2":
            draws = helper.sample_draws_from_quantiles(forecast, qt_levels, n_draws=2000, seed=seed)
        else:
            draws = forecast.astype(np.float32)

        # Compute probabilistic peak-time forecast (W, H, 1)
        peak_probs = helper.compute_windowed_inci_peak(draws, obs_win, P_peak)

        one_w, two_w = helper.split_week_forecast(peak_probs, start_indices, incidence.index, window_len=14, stride_size=7)
        one_w.to_csv(f"data/4_peak_time_predictions/{model}/{state}_1w_forecast.csv")
        two_w.to_csv(f"data/4_peak_time_predictions/{model}/{state}_2w_forecast.csv")


In [ ]:
def plot_peak_forecast_with_observed_peaks(
    state: str,
    horizon: str,              # "1w" or "2w"
    model_name: str,           # just for the legend/title
    pred_dir: str,             # directory containing the new model's forecasts
    peak_col_1w: str = "peak_prob_1w",
    peak_col_2w: str = "peak_prob_2w",
    test_dir: str = "data/test data",
):
    """
    Plot 1w or 2w peak-time forecasts for a selected state against time,
    with vertical lines marking observed peak dates.

    Assumptions:
      - Prediction file path is: {pred_dir}/{state}_{horizon}_forecast.csv
      - Test file path is:       {test_dir}/{state}_test.csv
      - Test CSV has a 'peak' column (0/1).
      - Prediction CSV has a single column with peak probabilities
        (name given by peak_col_1w or peak_col_2w).
    """

    pred_path = Path(pred_dir) / f"{state}_{horizon}_forecast.csv"
    test_path = Path(test_dir) / f"{state}_test.csv"

    # --- Load forecast ---
    pred_df = pd.read_csv(pred_path, index_col=0, parse_dates=True)
    if horizon == "1w":
        # unify column name
        pred_df = pred_df.rename(columns={peak_col_1w: "peak_prob"})
    elif horizon == "2w":
        pred_df = pred_df.rename(columns={peak_col_2w: "peak_prob"})
    else:
        raise ValueError("horizon must be '1w' or '2w'")

    # --- Load observed peaks ---
    test_df = pd.read_csv(test_path, index_col=0, parse_dates=True)
    # ensure column name is 'peak'
    if "peak" not in test_df.columns:
        # if your test data uses a different name, adapt here
        raise KeyError("Expected a 'peak' column in the test data.")

    # --- Align by date ---
    merged = pred_df.join(test_df[["peak"]], how="inner")
    merged = merged.sort_index()

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 4), dpi=140)

    # forecast time series
    ax.plot(
        merged.index,
        merged["peak_prob"],
        label=f"{model_name} {horizon} peak forecast",
    )

    ax.set_xlabel("Date")
    ax.set_ylabel("Peak probability")
    ax.set_title(f"{model_name} {horizon} peak-time forecast for {state.upper()}")

    # vertical lines at observed peaks
    peak_dates = merged.index[merged["peak"] == 1]
    first = True
    for d in peak_dates:
        ax.axvline(d, color="green", linestyle="--", alpha=0.5,
                   label="Observed peak" if first else None)
        first = False

    ax.legend(loc="upper left", frameon=False)
    ax.grid(alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_peak_forecast_with_observed_peaks(
    state="ny",
    horizon="2w",
    model_name="5 near peak",
    pred_dir="data/peak time predictions/5 near peak"
)